# Compare Rule-Based Scoring vs ML Baseline

この notebook では、同じ fixture feature rows を使って、rule-based scoring と logistic regression baseline を比較する。

目的は、高度な ML 性能を出すことではなく、ML model も `feature row -> risk_score` の関数として既存の evaluation harness に載せられることを確認すること。

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from abuse_detection.evaluation import evaluate_feature_rows
from abuse_detection.error_analysis import false_negatives, false_positives
from abuse_detection.ml_baseline import train_ml_baseline
from abuse_detection.scoring import scoring_fn

## Load Feature Rows

`label_value` は学習と評価に使う。scoring 時には、rule-based でも ML-based でも特徴量だけから `risk_score` を出す。

In [ ]:
feature_rows = pd.read_csv(PROJECT_ROOT / "fixtures" / "feature_rows_sample.csv")
feature_rows.head()

## Train ML Baseline

ここでは最小 baseline として、同じ fixture で logistic regression を学習する。これは性能評価としては楽観的になりやすいが、まず評価パイプラインの形を学ぶために使う。

In [ ]:
ml_model = train_ml_baseline(feature_rows)

## Compare Threshold Metrics

rule-based scoring と ML-based scoring を同じ threshold sweep で比較する。

In [ ]:
rule_scored, rule_metrics = evaluate_feature_rows(feature_rows, score_fn=scoring_fn)
ml_scored, ml_metrics = evaluate_feature_rows(feature_rows, score_fn=ml_model.score_row)

comparison = (
    rule_metrics.assign(score_source="rule_based")
    .pipe(lambda df: pd.concat([df, ml_metrics.assign(score_source="ml_baseline")]))
    .sort_values(["threshold", "score_source"])
    .reset_index(drop=True)
)
comparison

## Compare False Positives and False Negatives

同じ threshold で、どちらの scoring がどの行を誤判定したかを観察する。

In [ ]:
selected_threshold = 80

rule_fp = false_positives(rule_scored, threshold=selected_threshold).assign(score_source="rule_based")
ml_fp = false_positives(ml_scored, threshold=selected_threshold).assign(score_source="ml_baseline")
pd.concat([rule_fp, ml_fp]).sort_values(["score_source", "risk_score"], ascending=[True, False])

In [ ]:
rule_fn = false_negatives(rule_scored, threshold=selected_threshold).assign(score_source="rule_based")
ml_fn = false_negatives(ml_scored, threshold=selected_threshold).assign(score_source="ml_baseline")
pd.concat([rule_fn, ml_fn]).sort_values(["score_source", "risk_score"], ascending=[True, False])

## Notes

* ML baseline も `score_fn` として `evaluate_feature_rows` に渡せる。
* 今回は train / validation を分けていないため、性能値は学習用の参考値として読む。
* 次に進むなら、train / validation split、negative sampling、score calibration を分けて考える。